<a href="https://colab.research.google.com/github/Aswanth0704/gpu-programming-cpp/blob/main/SerialvsParallel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

if os.getenv("COLAB_RELEASE_TAG"): # If running in Google Colab:
  !mkdir -p Sources
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.05-Serial-vs-Parallel/Sources/ach.h -nv -O Sources/ach.h

2026-09-26 14:45:10 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.05-Serial-vs-Parallel/Sources/ach.h [4942/4942] -> "Sources/ach.h" [1]


## Segmented Sum:
- let's calculate the sum of temperatures per row
- A segmented sum is defined as taking a single input array and given a segment size, calculating the sum of each segment.
- Hence, this row-based sum is segmented problem.

## Tabulate algorithm:
The tabulate algorithm receives a sequence and a function. It then applies this function to index of each element in the sequence, and stores the result into the provided sequence.


In [ ]:
## example code
%%writefile Sources/tabulate.cpp

// skipping all the necessary includes just for the sake simplicity:
thrust::universal_vector<int> vec(4); // creates an empty thrust vector of size 4.
thrust::tabulate(
  thrust::device, vec.begin(), vec.end(), // here I want to store the tranformed indices to the vec itself
  [] __host__ __device__ (int index)->{
    return index*2; // now vec stores {0, 2, 4, 6}
  }
);

// not compiling the code.

Writing some sample code to save the segmented sums using thrust::tabulate. Reduction is a memory bound algorithm. Instead of analyzing its performance in terms of elapsed time, we could take a look at how many bytes does our implementation process in a second. This metric is called *achieved througput*


In [2]:
%%writefile Sources/naive-segmented-sum.cpp
#include <cstdio>
#include <chrono> // for date, time measurements
#include <thrust/tabulate.h>
#include <thrust/execution_policy.h>
#include <thrust/universal_vector.h>

// return sums in every row:
thrust::universal_vector<float> row_temperatures(
                                  int height,
                                  int width,
                                  const thrust::universal_vector<float> &temp)
{
  // raw pointer to temp:
  const float *d_temp_ptr = thrust::raw_pointer_cast(temp.data());
  thrust::universal_vector<float> sums(height);
  thrust::tabulate(
    thrust::device, sums.begin(), sums.end(),
    [=]__host__ __device__(int row_id)
    {
      float sum =0;
      for(int i = 0; i<width; i++)
      {
        sum+= d_temp_ptr[row_id*width + i];
      }
      return sum; // this value will be stored in sums
    }
  );

  return sums;
}

// return a sample thrust universal vector with values high and low of dimensions height and width.

thrust::universal_vector<float> init(int height, int width)
{

  const float low = 15.0;
  const float high = 90.0;
  thrust::universal_vector<float> temp(height*width, low);
  thrust::fill(thrust::device, temp.begin(), temp.begin()+width, high);
  // basically we are saying there is a hot boundary . only first row elements are 90
  return temp;
}

int main()
{
  int height = 16;
  int width = 16777216;
  thrust::universal_vector<float> temp = init(height, width);
  auto begin = std::chrono::high_resolution_clock::now();
  thrust::universal_vector<float> sums = row_temperatures(height, width, temp);
  auto end = std::chrono::high_resolution_clock::now();
  const double seconds = std::chrono::duration<double>(end- begin).count();
  const double gigabytes = static_cast<double>(temp.size()*sizeof(float))/1024/1024/1024;
  const double throughput = gigabytes/seconds;
  std::printf("computed in : %g seconds\n", seconds);
  std::printf("Throughput: %g GB/s\n", throughput);
  return 0;
}


Writing Sources/naive-segmented-sum.cpp


In [3]:
!nvcc --extended-lambda -o /tmp/a.out Sources/naive-segmented-sum.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

computed in : 0.573152 seconds
Throughput: 1.74474 GB/s


## Reduce by key
- For loop in the thrust::tabulate is executed sequentially. It could only process each of the 16 elements in parallel, while the operator processes over 16 million elements.
- To fix this, using thrust::reduce_by_key algorithm. Instead of reducing the sequence into a single value, it allows you to reduce segments of values. To distinguish these segments, we need to provide keys. Consecutive keys that are equal form a segment.


In [ ]:
## example
%%writefile Sources/example_reduce_by_key.cpp

// skip the includes
int in_keys[] = {1, 1, 1, 3, 3};
int in_vals[] = {1, 2, 3, 4, 5};
int out_keys[2];
int out_vals[2];

thrust::reduce_by_key(in_keys, in_keys+5, out_keys, out_vals);
// out_keys = {1, 3}
// out_vals = {6, 9}



Let;s frame segmented sum in reduce by key.

- we will first use thrust::make_transform_iterator and thrust::make_counting_iterators to create our row_ids
- and then use thrust::reduce_by_key to get segmented sums:

In [ ]:
%%writefile Sources/example_transform_iterator_on_counting_iterator.cpp
// skip include and execution, this is just for how to:
int constant = 2;
auto tranform = thrust::make_transform_iterator(
  // iterator to the beginning of the input sequence,
  vector.begin(),
  // capture the const in the lambda by value with `[name]`
  [constant]__host__ __device__(float value_from_input_sequence)
  {
    // transformation of each element
    return value_from_input_sequence * constant;

  }
);

// Example of counting iterator:
auto count_it = thrust::make_counting_iterator(0);




In [20]:
%%writefile Sources/reduce_by_key.cpp

#include <cstdio>
#include <chrono> // for date, time measurements

#include <thrust/execution_policy.h>
#include <thrust/universal_vector.h>
#include <thrust/reduce.h>
#include <thrust/fill.h>
#include <thrust/iterator/transform_iterator.h>
#include <thrust/iterator/counting_iterator.h>
#include <thrust/iterator/discard_iterator.h>



thrust::universal_vector<float> row_temperatures(
  int height, int width, const thrust::universal_vector<float> &temp)

  {
    thrust::universal_vector<float> sums(height);

    auto row_ids_begin = thrust::make_transform_iterator(
      thrust::make_counting_iterator(0),
      [=] __host__ __device__ (int id){
        return id/width;
      }
    );

    auto row_ids_end = row_ids_begin + temp.size();
    thrust::reduce_by_key(
      thrust::device,
      row_ids_begin, row_ids_end,
      temp.begin(),
      thrust::make_discard_iterator(),
      sums.begin()
    );
    return sums;
  }


  thrust::universal_vector<float> init(int height, int width) {
  const float low = 15.0;
  const float high = 90.0;
  thrust::universal_vector<float> temp(height * width, low);
  thrust::fill(thrust::device, temp.begin(), temp.begin() + width, high);
  return temp;
}

int main()
{
  int height = 16;
  int width = 16777216;
  thrust::universal_vector<float> temp = init(height, width);
  auto begin = std::chrono::high_resolution_clock::now();
  thrust::universal_vector<float> sums = row_temperatures(height, width, temp);
  auto end = std::chrono::high_resolution_clock::now();
  const double seconds = std::chrono::duration<double>(end- begin).count();
  const double gigabytes = static_cast<double>(temp.size()*sizeof(float))/1024/1024/1024;
  const double throughput = gigabytes/seconds;
  std::printf("computed in : %g seconds\n", seconds);
  std::printf("Throughput: %g GB/s\n", throughput);
  return 0;
}

Overwriting Sources/reduce_by_key.cpp


In [21]:
!nvcc --extended-lambda -o /tmp/a.out Sources/reduce_by_key.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

computed in : 0.00852695 seconds
Throughput: 117.275 GB/s


that's 67 times improvement in throughput

## Excercise Segmented mean!
## we need to modify the output itself using the thrust::make_transform_output_iterator

In [ ]:
%%writefile Sources/make_transform_output_iterator_example.cpp

// skip includes and execution
struct functor
{
  __host__ __device__
  float operator()(float value) const {
    // will store value/ 2 in the output

    return value/2;
  }

}

auto transform_output_iterator = thrust::make_tranform_output_iterator(
  // iterator to the beginning of the output sequence
  vector.begin(),
  // functor to apply before it's written to the vector.
  functor{}
)